# All-in-One AI Backend: Chat, TTS & Image Generation

This notebook turns a free Google Colab GPU (T4) into a multi-capability AI server exposing 3 FastAPI endpoints via ngrok:
1. **Chat Completions** (`/chat`): Powered by **Qwen2.5-1.5B-Instruct** (or compatible with OpenAI format)
2. **Text-to-Speech** (`/tts`): Powered by **facebook/mms-tts-eng** (VITS model returning high-fidelity WAV audio bytes)
3. **Image Generation** (`/generate-image`): Powered by **Stable Diffusion 1.5** returning PNG image bytes

---

## 📋 Quickstart Instructions

1. **Enable GPU Runtime**:
   - In the top menu, go to **Runtime** > **Change runtime type**.
   - Select **T4 GPU** under *Hardware accelerator* and click **Save**.

2. **Run Step 1 (Install Dependencies)**:
   - Execute the first code cell to install the required libraries (`diffusers`, `transformers`, `accelerate`, `fastapi`, `uvicorn`, `pyngrok`, `nest-asyncio`, `scipy`).

3. **Set Your ngrok Token in Step 2**:
   - Get your free authtoken from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken).
   - Replace `YOUR_NGROK_AUTH_TOKEN_HERE` with your actual token in **Step 2**.

4. **Run Step 2 (Load Models & Launch Server)**:
   - Execute the second code cell. Colab will download the models into GPU memory (~6.5 GB VRAM total, fitting comfortably within the 15 GB T4 limit).

5. **Retrieve Your Public ngrok URL**:
   - Copy the public URL printed in the output banner:
     ```text
     Public ngrok URL: https://xxxx-xx-xx-xx-xx.ngrok-free.app
     - Chat:  POST https://xxxx-xx-xx-xx-xx.ngrok-free.app/chat
     - TTS:   POST https://xxxx-xx-xx-xx-xx.ngrok-free.app/tts
     - Image: POST https://xxxx-xx-xx-xx-xx.ngrok-free.app/generate-image
     ```
   - Paste this URL into your local `multimodal_client.py` and run it!

In [ ]:
# Step 1: Install required packages on Google Colab
!pip install diffusers transformers accelerate fastapi uvicorn pyngrok nest-asyncio scipy

### Step 2: Load Models, Configure FastAPI, and Launch ngrok Tunnel
Make sure to paste your ngrok authtoken below before running.

In [ ]:
import io
from typing import List, Optional
import numpy as np
import scipy.io.wavfile as wavfile
import torch
from diffusers import StableDiffusionPipeline
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel
from pyngrok import ngrok
import nest_asyncio
import uvicorn
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    VitsModel,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[+] Target compute device: {device.upper()}")

# =============================================================================
# 1. MODEL INITIALIZATION
# =============================================================================

# --- Model A: Chat LLM (Qwen2.5-1.5B-Instruct) ---
print("[1/3] Loading Chat LLM (Qwen/Qwen2.5-1.5B-Instruct)...")
llm_id = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_id)
if llm_tokenizer.pad_token_id is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token
    llm_tokenizer.pad_token_id = llm_tokenizer.eos_token_id
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
print("    ✓ Chat LLM loaded.")

# --- Model B: Text-to-Speech (facebook/mms-tts-eng) ---
print("[2/3] Loading Text-to-Speech Model (facebook/mms-tts-eng)...")
tts_id = "facebook/mms-tts-eng"
tts_tokenizer = AutoTokenizer.from_pretrained(tts_id)
tts_model = VitsModel.from_pretrained(tts_id).to(device)
print("    ✓ TTS model loaded.")

# --- Model C: Image Generation (Stable Diffusion 1.5) ---
print("[3/3] Loading Image Generation Model (runwayml/stable-diffusion-v1-5)...")
sd_id = "runwayml/stable-diffusion-v1-5"
sd_pipe = StableDiffusionPipeline.from_pretrained(
    sd_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
sd_pipe = sd_pipe.to(device)
print("    ✓ Stable Diffusion pipeline loaded.")
print("[+] All 3 models ready in VRAM!")

# =============================================================================
# 2. FASTAPI APP & REQUEST SCHEMAS
# =============================================================================
app = FastAPI(title="Multimodal AI Colab Backend")

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    prompt: Optional[str] = None
    messages: Optional[List[ChatMessage]] = None
    max_tokens: int = 512
    temperature: float = 0.7

class TTSRequest(BaseModel):
    text: str

class ImageRequest(BaseModel):
    prompt: str
    num_inference_steps: int = 30
    guidance_scale: float = 7.5

# --- Endpoint 1: Chat Completions ---
@app.post("/chat")
@app.post("/v1/chat/completions")
def chat_endpoint(req: ChatRequest):
    try:
        if req.messages:
            formatted_messages = [{"role": m.role, "content": m.content} for m in req.messages]
        elif req.prompt:
            formatted_messages = [{"role": "user", "content": req.prompt}]
        else:
            raise HTTPException(status_code=400, detail="Provide either 'messages' or 'prompt'.")

        prompt_text = llm_tokenizer.apply_chat_template(
            formatted_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = llm_tokenizer([prompt_text], return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = llm_model.generate(
                **inputs,
                max_new_tokens=req.max_tokens,
                temperature=req.temperature if req.temperature > 0 else None,
                do_sample=req.temperature > 0,
                pad_token_id=llm_tokenizer.pad_token_id,
            )

        input_len = inputs.input_ids.shape[1]
        generated_ids = outputs[0][input_len:]
        reply = llm_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        return {
            "reply": reply,
            "choices": [{"message": {"role": "assistant", "content": reply}}],
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- Endpoint 2: Text-to-Speech ---
@app.post("/tts")
@app.post("/v1/audio/speech")
def tts_endpoint(req: TTSRequest):
    try:
        inputs = tts_tokenizer(req.text, return_tensors="pt").to(device)
        with torch.no_grad():
            waveform = tts_model(**inputs).waveform

        # Extract audio waveform and convert to 16-bit PCM WAV
        audio_data = waveform.squeeze().cpu().numpy()
        scaled = np.clip(audio_data, -1.0, 1.0)
        int16_pcm = (scaled * 32767).astype(np.int16)

        wav_buffer = io.BytesIO()
        wavfile.write(wav_buffer, rate=tts_model.config.sampling_rate, data=int16_pcm)
        wav_buffer.seek(0)

        return Response(content=wav_buffer.getvalue(), media_type="audio/wav")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- Endpoint 3: Image Generation ---
@app.post("/generate-image")
@app.post("/generate")
def image_endpoint(req: ImageRequest):
    try:
        image = sd_pipe(
            req.prompt,
            num_inference_steps=req.num_inference_steps,
            guidance_scale=req.guidance_scale,
        ).images[0]

        img_buffer = io.BytesIO()
        image.save(img_buffer, format="PNG")
        img_buffer.seek(0)

        return Response(content=img_buffer.getvalue(), media_type="image/png")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# =============================================================================
# 3. NGROK TUNNEL CONFIGURATION & UVICORN SERVER
# =============================================================================
# Paste your ngrok authtoken here:
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url
print("=" * 65)
print(f"🚀 Public ngrok Base URL: {public_url}")
print(f"   1. Chat Endpoint:       {public_url}/chat")
print(f"   2. Text-to-Speech:      {public_url}/tts")
print(f"   3. Image Generation:    {public_url}/generate-image")
print("=" * 65)

# Apply nest_asyncio to run uvicorn inside Colab notebook event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()
